# 02 Clinical Metadata EDA

Purpose:
- understand metadata quality and schema
- inspect PHQ-9, HAMD, and demographic fields
- prepare for severity-estimation target design

Primary questions:
- Which columns are available and complete?
- How much missingness exists?
- Are PHQ-9 and HAMD usable as targets?
- What severity-binning strategies are reasonable?


In [ ]:
from pathlib import Path
import pandas as pd

# Fixed metadata location under backend/data
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'backend' / 'data'
METADATA_DIR = DATA_DIR / 'metadata'
print('Using metadata dir:', METADATA_DIR)
METADATA_DIR

In [ ]:
# Clinical metadata EDA
import sys
from pathlib import Path
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
except Exception as e:
    print('Some plotting or pandas packages are missing:', e)
    pd = None

# Locate metadata workbook under backend/data
meta_candidates = []
if METADATA_DIR.exists():
    meta_candidates += list(METADATA_DIR.glob('*.xlsx')) + list(METADATA_DIR.glob('*.csv'))
# fallback to backend/data root workbook
root_xlsx = Path.cwd().resolve() / 'backend' / 'data' / 'subjects_information_audio_lanzhou_2015.xlsx'
if root_xlsx.exists():
    meta_candidates.insert(0, root_xlsx)

if not meta_candidates:
    print('No metadata workbook found under', METADATA_DIR)
else:
    meta_path = meta_candidates[0]
    print('Loading metadata from', meta_path)
    if pd is None:
        print('pandas not available; install backend[dev] to run metadata EDA')
    else:
        meta = pd.read_excel(meta_path) if meta_path.suffix in ('.xls', '.xlsx') else pd.read_csv(meta_path)
        display(meta.head())
        print('Rows, cols:', meta.shape)
        print('\nColumn types:')
        print(meta.dtypes)
        print('\nMissing values:')
        print(meta.isna().sum().sort_values(ascending=False).head(20))
\n
        # Try to find PHQ/HAMD columns (name variations)
        col_candidates = {
            'phq': [c for c in meta.columns if 'phq' in c.lower().replace('-', '')],
            'hamd': [c for c in meta.columns if 'hamd' in c.lower()],
        }
        print('Found candidate columns:', {k: v[:3] for k, v in col_candidates.items()})
\n
        # Analyze PHQ-9 if present
        phq_cols = col_candidates.get('phq', [])
        if phq_cols:
            phq_col = phq_cols[0]
            print('\nUsing PHQ column:', phq_col)
            s = pd.to_numeric(meta[phq_col], errors='coerce')
            print('Missing PHQ count:', s.isna().sum())
            display(s.describe())
            if 'plt' in globals():
                fig, ax = plt.subplots(1, 2, figsize=(12, 4))
                sns.histplot(s.dropna(), bins=15, ax=ax[0])
                sns.boxplot(x=s.dropna(), ax=ax[1])
                ax[0].set_title('PHQ distribution')
                ax[1].set_title('PHQ boxplot')
                plt.show()
            # severity bins (PHQ-9 standard): 0-4 none,5-9 mild,10-14 mod,15-19 mod-severe,20+ severe
            bins = [0, 5, 10, 15, 20, 100] 
            labels = ['none', 'mild', 'moderate', 'moderately_severe', 'severe']
            meta['_phq_severity'] = pd.cut(s.fillna(-1), bins=[-1]+bins, labels=['missing']+labels)
            print('\nPHQ severity counts:')
            display(meta['_phq_severity'].value_counts(dropna=False))
\n
        else:
            print('No PHQ candidate column found; consider column names in metadata workbook')
\n
        # HAMD if present (similar flow)
        hamd_cols = col_candidates.get('hamd', [])
        if hamd_cols:
            hamd_col = hamd_cols[0]
            print('\nUsing HAMD column:', hamd_col)
            s2 = pd.to_numeric(meta[hamd_col], errors='coerce')
            display(s2.describe())
        else:
            print('No HAMD column found')